## 1. Load & Inspect the Dataset

We load `psa_dataset_dholuo_somali.csv` and check its basic shape and health:
row/column counts, missing values, duplicates, and domain balance. This tells
us what preprocessing decisions are actually needed for this dataset, rather
than assuming the same issues Ekegusii had.

In [6]:
import pandas as pd

df = pd.read_csv("../data/processed/psa_dataset_dholuo_somali.csv", encoding="utf-8")

# Basic shape
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Missing values per column
print("\nMissing values:\n", df.isnull().sum())

# Duplicates
print("\nFully duplicate rows:", df.duplicated().sum())
print("Duplicate Dholuo strings:", df["Dholuo"].duplicated().sum())
print("Duplicate Somali strings:", df["Somali"].duplicated().sum())

# Domain balance
print("\nDomain counts:\n", df["Domain"].value_counts())

# Class / Source breakdown
print("\nClass counts:\n", df["Class"].value_counts())
print("\nSource counts:\n", df["Source"].value_counts())

df.head(3)

Shape: (16029, 8)

Columns: ['PSA_Id', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Somali', 'Class', 'Source']

Missing values:
 PSA_Id       0
Domain       0
English      0
Kiswahili    0
Dholuo       0
Somali       1
Class        0
Source       0
dtype: int64

Fully duplicate rows: 0
Duplicate Dholuo strings: 27
Duplicate Somali strings: 21

Domain counts:
 Domain
Health               3764
Agriculture          3645
Education            2844
Governance           2748
Security             1795
Security & Safety    1233
Name: count, dtype: int64

Class counts:
 Class
General    10917
PSA         5112
Name: count, dtype: int64

Source counts:
 Source
grounded_generated           10917
original_baseline_dataset     5112
Name: count, dtype: int64


,PSA_Id,Domain,English,Kiswahili,Dholuo,Somali,Class,Source
0,1,Education,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Chenro mag thieth: Chenro mag thieth kod ritru...,Barnaamijyada caafimaadka iyo amniga ee COVID-...,PSA,original_baseline_dataset
1,2,Education,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Kenya Education Cloud: Ohinga mar somo mar dij...,Barashada dhijitaalka ah ee bixisa waxyaabaha ...,PSA,original_baseline_dataset
2,3,Education,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,KUCCPS Portal: KUCCPS portal biro yawore e dwe...,Gudaha KUCCPS waxaa la furi doonaa bishii Maar...,PSA,original_baseline_dataset


In [9]:
missing_row = df[df["Somali"].isna()]
print(missing_row[["PSA_Id", "English","Kiswahili"]])  # see which row it is 

      PSA_Id                                            English  \
4888    4889  Citizens who understand accessing government s...   

                                              Kiswahili  
4888  Wananchi wanaoelewa kupata huduma za serikali ...  


## Interpretation: Initial Inspection

- **Size:** 16,029 rows across 8 columns (PSA_Id, Domain, English, Kiswahili,
  Dholuo, Somali, Class, Source).
- **Missing values:** Only 1 row (PSA_Id 4889, "Citizens who understand
  accessing government s...") is missing a Somali translation, out of
  16,029 — a negligible gap (0.006%), traced to a single failed request in
  the machine-translation script. Documented as an accepted, known minor
  gap rather than silently dropped.
- **Duplicates:** No fully duplicate rows. 27 duplicate Dholuo strings and
  21 duplicate Somali strings exist — worth a closer look in the next pass
  to confirm these are legitimate (e.g. reused short PSA phrases) rather
  than translation shortcuts, the way Ekegusii's notebook found for its
  own duplicates.
- **Domain balance:** Health (3,764) and Agriculture (3,645) are the
  largest domains; Education (2,844) and Governance (2,748) are mid-sized;
  Security and Security & Safety appear as two separate categories
  totaling 1,795 + 1,233 — this needs a team decision on whether to merge
  them, since it affects the true domain balance picture.
- **Class/Source:** 10,917 rows are grounded_generated (Class=General),
  5,112 are original_baseline_dataset (Class=PSA) — a roughly 2:1 synthetic-
  to-original ratio, consistent with the dataset's stated construction.

## 2. Quantifying Orthographic & Encoding Issues

Before normalizing anything, we count known issue types across English,
Kiswahili, Dholuo, and Somali columns: mojibake (encoding corruption),
smart-quote/apostrophe variants, and stray bracket artifacts. Dholuo uses
apostrophes for specific phonemes (e.g. ng'), so we check apostrophe
*variants* (straight vs curly), not apostrophes themselves, since removing
them incorrectly would damage real orthography.

In [10]:
import re

def count_pattern(df, cols, pattern, label):
    print(f"\n--- {label} ---")
    for col in cols:
        matches = df[col].astype(str).str.contains(pattern, regex=True, na=False)
        print(f"{col}: {matches.sum()} rows")

text_cols = ["English", "Kiswahili", "Dholuo", "Somali"]

# Mojibake — classic UTF-8/Windows-1252 double-encoding markers
count_pattern(df, text_cols, r"Â|â€|Ã©|Ã¢", "Mojibake indicators")

# Smart quote variants (curly quotes/apostrophes)
count_pattern(df, text_cols, r"[’‘“”]", "Curly quote/apostrophe variants")

# Straight apostrophe count for comparison (expected/legitimate in Dholuo, e.g. ng')
count_pattern(df, text_cols, r"'", "Straight apostrophes")

# Stray brackets / leftover markup artifacts
count_pattern(df, text_cols, r"[\[\]{}]", "Stray brackets")

# Double spaces / leading-trailing whitespace
count_pattern(df, text_cols, r"  +", "Double+ spaces")
whitespace_issues = df[text_cols].apply(lambda col: col.astype(str).str.strip() != col.astype(str))
print("\n--- Leading/trailing whitespace ---")
print(whitespace_issues.sum())


--- Mojibake indicators ---
English: 5 rows
Kiswahili: 7 rows
Dholuo: 0 rows
Somali: 2 rows

--- Curly quote/apostrophe variants ---
English: 0 rows
Kiswahili: 0 rows
Dholuo: 8215 rows
Somali: 264 rows

--- Straight apostrophes ---
English: 3446 rows
Kiswahili: 117 rows
Dholuo: 2768 rows
Somali: 2498 rows

--- Stray brackets ---
English: 8 rows
Kiswahili: 8 rows
Dholuo: 9 rows
Somali: 8 rows

--- Double+ spaces ---
English: 0 rows
Kiswahili: 0 rows
Dholuo: 0 rows
Somali: 0 rows

--- Leading/trailing whitespace ---
English      0
Kiswahili    0
Dholuo       0
Somali       0
dtype: int64


## Interpretation: Orthographic Noise

- **Mojibake:** Minimal across the board — English (5), Kiswahili (7),
  Somali (2), Dholuo (0). Not a significant issue for this dataset; the
  multi-pass ftfy pipeline Ekegusii needed is likely overkill here. A
  single light pass or even manual spot-check of these 14 rows is
  sufficient.
- **Curly quote/apostrophe variants:** This is the dataset's real finding.
  Dholuo has 8,215 rows (over half the dataset) with curly apostrophes/
  quotes, and Somali has 264. English and Kiswahili have zero. This
  strongly suggests the Dholuo column was passed through a process (likely
  MT or copy-paste from a formatted source) that converted straight
  apostrophes into typographic/curly ones inconsistently.
- **Straight apostrophes:** Present across all four columns (English 3,446;
  Dholuo 2,768; Somali 2,498; Kiswahili 117), meaning Dholuo currently has
  a *mix* of both curly and straight apostrophe styles for what should be
  the same linguistic feature — likely the ng' consonant marker and
  similar orthographic apostrophes. This is inconsistency, not necessarily
  incorrect content.
- **Stray brackets:** Minor and roughly even across columns (8-9 rows
  each) — likely leftover scraping artifacts, low priority.
- **Decision:** Standardize all curly apostrophe/quote variants (’ ‘ “ ”)
  to their straight equivalents (' ") across all four text columns, so
  that Dholuo's orthographic apostrophe usage (e.g. ng') is represented
  consistently rather than split across two visually different but
  linguistically identical characters. This is a normalization
  (consistency) fix, not a content removal — the apostrophes themselves
  are correct Dholuo orthography and must be preserved, only their
  character encoding is being standardized.